# Experiment 2 revised pilot — GPU qualification

Development-only check pinned to `cc29e21ebb307e9b3d65b32a89edf78a81a9b0ae` on `codex/exp2-revised-pilot-20261112`. Verifies the four isolated policy branches, frozen versus learning critics, exact 10,000-raw-step accounting (shortened here to 20), the plasticity probe, interrupted restart, both reference regimes at full critic size, and the two-runs-per-T4 VRAM ceiling. A pass here does **not** qualify the scientific 1M-step population and does not authorize results.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ.update({"JAX_PLATFORMS": "cuda,cpu", "MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl", "EGL_PLATFORM": "surfaceless", "MUJOCO_EGL_DEVICE_ID": "0", "WANDB_MODE": "disabled", "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "XLA_PYTHON_CLIENT_PREALLOCATE": "false"})
import signal
signal.signal(signal.SIGALRM, lambda signum, frame: (_ for _ in ()).throw(TimeoutError("Experiment 2 revised pilot qualification exceeded its limit")))
signal.alarm(4 * 60 * 60)
import platform
import shutil
import socket
import subprocess
import sys
import tempfile
import time
from pathlib import Path

SOURCE_SHA = "cc29e21ebb307e9b3d65b32a89edf78a81a9b0ae"
SOURCE_BRANCH = "codex/exp2-revised-pilot-20261112"
REPO = Path("/kaggle/temp/scaling-drl-research")
REPO.parent.mkdir(parents=True, exist_ok=True)

def wait_for_dns(host, attempts=6, delay=10):
    for attempt in range(attempts):
        try:
            socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
            print("DNS available:", host)
            return
        except socket.gaierror as exc:
            print(f"DNS attempt {attempt + 1}/{attempts} failed for {host}: {exc}", flush=True)
            if attempt + 1 < attempts:
                time.sleep(delay)
    raise RuntimeError(f"Kaggle cannot resolve {host}. Enable Internet and retry.")

for host in ("github.com", "pypi.org", "files.pythonhosted.org"):
    wait_for_dns(host)

if not REPO.exists():
    for attempt in range(3):
        with tempfile.TemporaryDirectory(prefix="exp2-clone-", dir=REPO.parent) as tmp:
            destination = Path(tmp) / "source"
            result = subprocess.run(["git", "clone", "--quiet", "--single-branch", "--branch", SOURCE_BRANCH, "https://github.com/kaves29/scaling-drl-research.git", str(destination)], text=True, capture_output=True)
            if result.returncode == 0:
                shutil.move(str(destination), str(REPO))
                break
            print(f"Git clone attempt {attempt + 1}/3 failed: {result.stderr.strip()}", flush=True)
        if attempt < 2:
            time.sleep(10)
    else:
        raise RuntimeError("Could not clone the pinned source after three attempts.")
subprocess.run(["git", "fetch", "--quiet", "origin", SOURCE_SHA], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_SHA], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == SOURCE_SHA

requirements = (REPO / "main/requirements.txt").read_text()
install_dm_control_without_optional_mazes = sys.version_info >= (3, 13)
if install_dm_control_without_optional_mazes:
    replacements = {"numpy==1.26.4": "numpy==2.1.3", "pandas==2.1.4": "pandas==2.2.3", "matplotlib==3.9.0": "matplotlib==3.9.2", "Shimmy==2.0.0": "Shimmy==1.3.0"}
    for old, new in replacements.items():
        assert requirements.count(old) == 1, f"Expected exactly one {old} pin"
        requirements = requirements.replace(old, new)
    for unused_pin in ("labmaze==1.0.6", "dm_control==1.0.38"):
        assert requirements.count(unused_pin) == 1, f"Expected exactly one {unused_pin} pin"
        requirements = requirements.replace(unused_pin + "\n", "")
runtime_requirements = Path("/kaggle/temp/exp2-runtime-requirements.txt")
runtime_requirements.write_text(requirements)
print("Python runtime:", platform.python_version(), "; installing:", runtime_requirements)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(runtime_requirements)], check=True)
if install_dm_control_without_optional_mazes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "dm_control==1.0.38"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "jax-cuda13-plugin", "jax-cuda13-pjrt"], check=True, stdout=subprocess.DEVNULL)
    print("Removed Kaggle CUDA 13 JAX plugins to avoid conflict with the pinned CUDA 12 backend.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "jax[cuda12]==0.4.34"], check=True)

import importlib.metadata as package_metadata
import flax
import gymnasium
import jax
import matplotlib
import mujoco
import numpy
if not hasattr(numpy, "issctype"):
    def _numpy_issctype(rep):
        return isinstance(rep, type) and issubclass(rep, numpy.generic)
    numpy.issctype = _numpy_issctype
import optax
import pandas
from dm_control import suite as dmc_suite
assert ("dog", "run") in dmc_suite.ALL_TASKS
print("Core runtime:", {name: package_metadata.version(name) for name in ("jax", "numpy", "flax", "optax", "dm-control", "gymnasium", "mujoco")})
print("Pinned revised-pilot source installed:", SOURCE_SHA)


In [ ]:
# Set rendering and JAX selection before importing either library.
os.environ.update({
    "JAX_PLATFORMS": "cuda,cpu", "MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl",
    "EGL_PLATFORM": "surfaceless", "MUJOCO_EGL_DEVICE_ID": "0", "WANDB_MODE": "disabled",
    "XDG_CACHE_HOME": "/kaggle/temp/.cache", "MPLCONFIGDIR": "/kaggle/temp/.mpl",
    "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "PYTHONDONTWRITEBYTECODE": "1",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
})
import jax
import numpy as np
names = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).splitlines()
assert len(names) == 2 and all("T4" in name for name in names), f"Expected 2 T4 GPUs, got {names}"
devices = jax.devices("gpu")
assert jax.default_backend() == "gpu" and len(devices) == 2, devices
for device in devices:
    with jax.default_device(device):
        value = jax.device_put(np.ones((128, 128), dtype=np.float32), device)
        out = jax.jit(lambda array: array @ array)(value)
        out.block_until_ready()
        assert out.devices() == {device}
print("GPU inventory:", names)
print("Verified JAX dispatch on both T4 devices:", devices)

import threading
class _Heartbeat:
    """Keep nbclient's 4-second IOPub timer alive during quiet long cells."""
    def __init__(self, label, interval=2):
        self.label, self.interval, self.count = label, interval, 0
        self.done = threading.Event()
        self.thread = threading.Thread(target=self._beat, daemon=True)
        self.thread.start()
    def _beat(self):
        while not self.done.wait(self.interval):
            self.count += 1
            print(f"[heartbeat] {self.label} {self.count}", flush=True)
    def stop(self):
        self.done.set()
        self.thread.join(timeout=1)

def heartbeat(label):
    return _Heartbeat(label)


In [ ]:
import subprocess
beat = heartbeat("contracts")
# Run the CPU contracts in a subprocess so heavy JAX/warning output cannot
# drown the IOPub channel that nbclient is watching.
CONTRACT_CODE = """
import numpy
if not hasattr(numpy, "issctype"):
    numpy.issctype = lambda rep: isinstance(rep, type) and issubclass(rep, numpy.generic)
import sys, unittest
names = ["tests.test_revised_exp2_pilot", "tests.test_online_exp2",
         "tests.test_online_exp2_diagnostic_primitives"]
suite = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromName(n) for n in names)
result = unittest.TextTestRunner(verbosity=2).run(suite)
print("RAN", result.testsRun, "FAILURES", len(result.failures), "ERRORS", len(result.errors))
sys.exit(0 if result.wasSuccessful() and result.testsRun >= 14 else 1)
"""
environment = dict(os.environ, PYTHONWARNINGS="ignore", JAX_PLATFORMS="cuda,cpu")
process = subprocess.Popen([sys.executable, "-c", CONTRACT_CODE], cwd=REPO / "main",
                           env=environment, text=True, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, bufsize=1)
summary = []
for line in process.stdout:
    summary.append(line.rstrip())
    if any(token in line for token in ("ok", "FAIL", "ERROR", "Ran ", "OK", "FAILED", "Traceback")):
        print(line.rstrip(), flush=True)
code = process.wait()
assert code == 0, "GPU source contracts failed; last lines:\n" + "\n".join(summary[-60:])
assert any(line.startswith("Ran ") for line in summary), summary[-20:]
sys.path.insert(0, str(REPO / "main"))
from experiments.exp12.revised_exp2_population import population
assert len(population()) == 20
help_result = subprocess.run(
    [sys.executable, str(REPO / "main/scripts/run_revised_exp2_pilot.py"), "--help"],
    cwd=REPO / "main", env=environment, text=True, capture_output=True)
assert help_result.returncode == 0, help_result.stderr[-3000:]
assert "run-index" in help_result.stdout
print("CPU/GPU contracts passed:", summary[-1], "; launch entry point imports")
beat.stop()


In [ ]:
import json
import time
import warnings
warnings.filterwarnings("ignore")
beat = heartbeat("gpu-smoke")
from unittest.mock import patch

from experiments.exp1 import compose_config
from experiments.online_exp2 import OnlineExp2
from experiments.exp12.revised_exp2_pilot import CONDITIONS, RevisedBranch, revised_analysis

RAW = 20
results, timings = {}, {}

def smoke_cfg(regime):
    overrides = [
        "online_exp2.environments=[dog-run]", "online_exp2.training_seeds=[1]",
        f"online_exp2.reference_regime={regime}",
        "online_exp2.training_interactions=3", "online_exp2.checkpoint_steps=[2]",
        "online_exp2.probe_checkpoints=[2]",
        f"online_exp2.diagnostic_raw_steps={RAW}", "online_exp2.diagnostic_save_every=10",
        "online_exp2.resource_cap_gpu_hours=1",
        "buffer.min_length=1", "buffer.sample_batch_size=8", "buffer.max_length=64",
        "probe.rounds=1", "probe.steps=2", "probe.pool_size=8",
        "probe.batch_size=8", "probe.eval_chunk=8",
        "logging_per_interaction_step=100", "evaluation_per_interaction_step=100",
        "num_eval_episodes=1", "actor_grad_cosine_every=100",
        "env.max_episode_steps=5", "env.action_repeat=2", "action_repeat=2",
        "wandb_enabled=false",
    ]
    return compose_config(str(REPO / "main/configs"), "online_exp2_revised_pilot", overrides, allow_pilot=True)

def run(out, regime, interrupt=False):
    fail = [interrupt]
    original = RevisedBranch.observe_and_update
    def interrupt_once(branch, timestep, observation):
        if fail[0] and branch.raw_steps >= 8:
            fail[0] = False
            raise RuntimeError("expected qualification interruption")
        return original(branch, timestep, observation)
    with patch("experiments.online_exp2.validate", return_value=(3, [2], [2])):
        if interrupt:
            with patch.object(RevisedBranch, "observe_and_update", interrupt_once):
                try:
                    OnlineExp2(smoke_cfg(regime), str(out)).run()
                except RuntimeError as exc:
                    assert str(exc) == "expected qualification interruption", exc
                else:
                    raise AssertionError("interruption did not occur")
        runner = OnlineExp2(smoke_cfg(regime), str(out))
        runner.run()
        return runner

for regime in ("reference_small", "reference_large"):
    print("PHASE reference training + probe + four branches:", regime, flush=True)
    out = Path("/kaggle/temp") / f"revised-exp2-{regime}"
    shutil.rmtree(out, ignore_errors=True)
    started = time.time()
    runner = run(out, regime, interrupt=(regime == "reference_small"))
    timings[regime] = time.time() - started
    records = [row for row in runner.rows if row.get("type") == "revised_pilot_checkpoint"]
    assert len(records) == 1, records
    record = records[0]
    assert set(record["branches"]) == set(CONDITIONS)
    for name, state in record["branches"].items():
        assert state["raw_steps"] == RAW, (name, state["raw_steps"])
        assert state["interactions"] == sum(e["length"] for e in record["branch_episodes"][name]), name
        assert 0 < state["optimizer_updates"], name
    for architecture in ("small", "large"):
        frozen = record["branches"][f"{architecture}_frozen"]
        learning = record["branches"][f"{architecture}_learning"]
        assert frozen["critic_hash"] == record["shadow_hashes"][architecture], architecture
        assert learning["critic_hash"] != record["shadow_hashes"][architecture], architecture
        assert frozen["actor_hash"] != learning["actor_hash"], architecture
    probes = [row for row in runner.rows if row.get("type") == "probe"]
    assert len(probes) == 2 and all(Path(row["artifact"]).exists() for row in probes), probes
    # Reference state must match the hashes recorded before branches ran.
    analysis = revised_analysis(runner)
    assert set(analysis["conditions"]) == set(CONDITIONS)
    assert len(runner.trainer.buffer) == 3
    assert all(shadow.updates == runner.trainer.update_step for shadow in runner.shadows.values())
    results[regime] = {"analysis": analysis, "record": record, "probe_rows": len(probes)}
    print("PHASE complete:", regime, flush=True)
    print("PASS full-size revised branch smoke:", regime,
          {name: (state["raw_steps"], state["interactions"], state["optimizer_updates"])
           for name, state in record["branches"].items()}, flush=True)

print("Wall-clock seconds:", timings)
Path("/kaggle/working/exp2_revised_qualification_summary.json").write_text(json.dumps(
    {"source_sha": SOURCE_SHA, "raw_steps_per_branch": RAW, "results": results, "timings": timings},
    indent=2, sort_keys=True))
print("PASS revised pilot GPU qualification (development schedule, not a scientific run)")
beat.stop()


In [ ]:
import subprocess
import time
import warnings
warnings.filterwarnings("ignore")
beat = heartbeat("vram-smoke")

def used_memory_mib():
    values = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        text=True).split()
    return [int(v) for v in values]

WORKER = """
import os
import sys
import numpy
if not hasattr(numpy, "issctype"):
    numpy.issctype = lambda rep: isinstance(rep, type) and issubclass(rep, numpy.generic)
from unittest.mock import patch
sys.path.insert(0, os.environ["EXP2_MAIN"])
from experiments.exp1 import compose_config
from experiments.online_exp2 import OnlineExp2
cfg = compose_config(os.environ["EXP2_CONFIG"], "online_exp2_revised_pilot", [
    "env_name=dog-run", "seed=2", "online_exp2.environments=[dog-run]",
    "online_exp2.training_seeds=[2]", "online_exp2.reference_regime=reference_small",
    "online_exp2.training_interactions=3", "online_exp2.checkpoint_steps=[2]",
    "online_exp2.probe_checkpoints=[]", "online_exp2.diagnostic_raw_steps=20",
    "online_exp2.diagnostic_save_every=10", "online_exp2.resource_cap_gpu_hours=1",
    "buffer.min_length=1", "buffer.sample_batch_size=8", "buffer.max_length=64",
    "probe.rounds=1", "probe.steps=2", "probe.pool_size=8", "probe.batch_size=8",
    "probe.eval_chunk=8", "logging_per_interaction_step=100",
    "evaluation_per_interaction_step=100", "num_eval_episodes=1",
    "actor_grad_cosine_every=100", "env.max_episode_steps=5",
    "env.action_repeat=2", "action_repeat=2", "wandb_enabled=false"], allow_pilot=True)
with patch("experiments.online_exp2.validate", return_value=(3, [2], [])):
    OnlineExp2(cfg, "/kaggle/temp/concurrency-smoke/" + os.environ["SMOKE_TAG"]).run()
print("CONCURRENCY-WORKER-DONE", os.environ["SMOKE_TAG"], flush=True)
"""

baseline = used_memory_mib()
processes = []
for tag in ("a", "b"):
    log_path = Path(f"/kaggle/temp/concurrency-{tag}.log")
    log = log_path.open("w")
    environment = dict(os.environ, CUDA_VISIBLE_DEVICES="0", SMOKE_TAG=tag,
                       EXP2_MAIN=str(REPO / "main"), EXP2_CONFIG=str(REPO / "main/configs"),
                       PYTHONWARNINGS="ignore", JAX_PLATFORMS="cuda,cpu",
                       WANDB_MODE="disabled", XLA_PYTHON_CLIENT_PREALLOCATE="false")
    processes.append((subprocess.Popen([sys.executable, "-c", WORKER], env=environment,
                                       text=True, stdout=log, stderr=subprocess.STDOUT),
                      log, tag, log_path))
peak = baseline[0]
while any(process.poll() is None for process, _, _, _ in processes):
    peak = max(peak, used_memory_mib()[0])
    time.sleep(5)
for process, log, tag, log_path in processes:
    log.close()
    assert process.returncode == 0, (tag, log_path.read_text()[-3000:])
    assert "CONCURRENCY-WORKER-DONE" in log_path.read_text(), log_path.read_text()[-1000:]
peak = max(peak, used_memory_mib()[0])
total_mib = int(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
    text=True).split()[0])
two_per_gpu = peak < 0.90 * total_mib
print(f"Peak GPU memory with two concurrent full-size runs: {peak} MiB of {total_mib} MiB")
print("Two runs per T4:", "FIT" if two_per_gpu else "DO NOT FIT; use one run per GPU")
Path("/kaggle/working/exp2_vram_smoke.txt").write_text(
    f"peak_mib={peak}\ntotal_mib={total_mib}\ntwo_per_gpu={two_per_gpu}\n")
beat.stop()
